In [7]:
import os
import pandas as pd

file_path = "/Users/dipachowdhury/Downloads/online_retail_II.xlsx"

print("File exists:", os.path.exists(file_path))

if os.path.exists(file_path):
    excel = pd.ExcelFile(file_path)
    print("Dataset sheets:", excel.sheet_names)
else:
    print("File not found. Check the file path.")


File exists: True
Dataset sheets: ['Year 2009-2010', 'Year 2010-2011']


In [9]:
%pip install pandas numpy openpyxl

Note: you may need to restart the kernel to use updated packages.


In [1]:
%%writefile customer_growth_analysis.py
from __future__ import annotations

import argparse
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

COLUMN_ALIASES = {
    "invoice": ["Invoice", "InvoiceNo"],
    "stock_code": ["StockCode"],
    "description": ["Description"],
    "quantity": ["Quantity"],
    "invoice_date": ["InvoiceDate"],
    "price": ["Price", "UnitPrice"],
    "customer_id": ["Customer ID", "CustomerID"],
    "country": ["Country"],
}
REQUIRED = ["invoice", "stock_code", "quantity", "invoice_date", "price", "customer_id"]
NON_PRODUCT_CODES = {"POST", "D", "M", "DOT", "BANK CHARGES", "CRUK", "AMAZONFEE", "S", "B", "ADJUST", "ADJUST2", "TEST001", "TEST002"}

SEGMENTS = ["Champions", "High-value at risk", "Active repeat buyers",
            "Recent one-time buyers", "Lapsed repeat buyers", "Lost one-time buyers"]
SEGMENT_ACTIONS = {
    "Champions": "RETAIN",
    "High-value at risk": "WIN BACK (priority)",
    "Active repeat buyers": "GROW",
    "Recent one-time buyers": "GROW (secure 2nd purchase)",
    "Lapsed repeat buyers": "WIN BACK (low-cost, automated)",
    "Lost one-time buyers": "DEPRIORITISE",
}


@dataclass
class Config:
    gross_margin: float = 0.30        # ASSUMPTION - dataset has no product cost data
    target_ltv_cac: float = 3.0       # ASSUMPTION - common rule of thumb
    inactive_days: int = 90           # no purchase for this long = inactive
    repeat_window_days: int = 90      # window for the adjusted repeat rate
    high_value_pct: float = 0.20      # top 20% of customers by spend = "high value"
    min_cohort_size: int = 30         # smaller cohorts are flagged low_sample
    min_product_customers: int = 100  # min first-day buyers for a product to be ranked
    value_horizon_months: int = 12    # observed value window per cohort
    winback_list_size: int = 200


# ----------------------------------------------------------------------------
# 1. LOAD / VALIDATE / CLEAN
# ----------------------------------------------------------------------------
def load(path: str | Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}. Download the dataset first (see README).")
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.concat(pd.read_excel(path, sheet_name=None).values(), ignore_index=True)
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="ISO-8859-1")


def standardise_columns(df: pd.DataFrame) -> pd.DataFrame:
    lower = {str(c).strip().lower(): c for c in df.columns}
    rename = {}
    for std, aliases in COLUMN_ALIASES.items():
        for a in aliases:
            if a.lower() in lower:
                rename[lower[a.lower()]] = std
                break
    df = df.rename(columns=rename)
    missing = [c for c in REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required column(s): {missing}. Columns found: {list(df.columns)}")
    for optional in ("description", "country"):
        if optional not in df.columns:
            df[optional] = ""
    return df[list(COLUMN_ALIASES.keys())].copy()


def clean(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    log = [("Raw rows", 0, len(df))]

    def record(step, before, after):
        log.append((step, before - after, after))

    n = len(df)
    df = df.copy()
    df["customer_id"] = pd.to_numeric(df["customer_id"], errors="coerce")
    df = df.dropna(subset=["customer_id"])
    record("Missing / invalid Customer ID (guest checkouts)", n, len(df)); n = len(df)

    df["invoice"] = df["invoice"].astype(str).str.strip()
    df = df[~df["invoice"].str.upper().str.startswith("C")]
    record("Cancellation invoices (start with 'C')", n, len(df)); n = len(df)

    df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
    df["price"] = pd.to_numeric(df["price"], errors="coerce")
    df = df[(df["quantity"] > 0) & (df["price"] > 0)]
    record("Non-positive quantity or price", n, len(df)); n = len(df)

    df["stock_code"] = df["stock_code"].astype(str).str.strip().str.upper()
    df = df[~df["stock_code"].isin(NON_PRODUCT_CODES)]
    record("Non-product codes (postage, fees, adjustments)", n, len(df)); n = len(df)

    df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
    df = df.dropna(subset=["invoice_date"])
    record("Unparseable invoice date", n, len(df)); n = len(df)

    df = df.drop_duplicates()
    record("Exact duplicate rows", n, len(df))

    if df.empty:
        raise ValueError("No usable rows left after cleaning. Check the input file.")
    df["customer_id"] = df["customer_id"].astype(int)
    df["revenue"] = df["quantity"] * df["price"]
    df["description"] = df["description"].fillna("").astype(str)
    df["country"] = df["country"].fillna("").astype(str)
    dq = pd.DataFrame(log, columns=["step", "rows_removed", "rows_remaining"])
    return df.reset_index(drop=True), dq


# ----------------------------------------------------------------------------
# 2. ORDERS (one row per invoice)
# ----------------------------------------------------------------------------
def build_orders(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse line items into one row per (customer, invoice)."""
    return (df.groupby(["customer_id", "invoice"], as_index=False)
              .agg(order_date=("invoice_date", "min"), revenue=("revenue", "sum"))
              .assign(revenue=lambda d: d["revenue"].round(2)))


def cohort_base(orders: pd.DataFrame) -> pd.DataFrame:
    """Per acquisition cohort: customers and revenue split into first-visit vs repeat.
    'Repeat' = an order on a later day than the customer's first purchase day.
    NOTE: older cohorts have had longer to repeat, so these cumulative totals are not
    comparable across cohorts; use the 12-month and window-adjusted metrics for comparisons."""
    g = orders.groupby("cohort_month")
    out = pd.DataFrame({
        "customers": g["customer_id"].nunique(),
        "total_revenue": g["revenue"].sum(),
        "first_visit_revenue": orders[orders["is_first_day_order"]].groupby("cohort_month")["revenue"].sum(),
        "repeat_revenue_to_date": orders[~orders["is_first_day_order"]].groupby("cohort_month")["revenue"].sum(),
        "ever_repeated_customers": orders[~orders["is_first_day_order"]].groupby("cohort_month")["customer_id"].nunique(),
    }).fillna(0)
    out["ever_repeated_customers"] = out["ever_repeated_customers"].astype(int)
    return out.round(2).reset_index()


def monthly_kpis(orders: pd.DataFrame) -> pd.DataFrame:
    """Monthly orders, customers, revenue, AOV and new-vs-returning revenue."""
    g = orders.groupby("order_month")
    out = pd.DataFrame({
        "orders": g.size(),
        "active_customers": g["customer_id"].nunique(),
        "revenue": g["revenue"].sum(),
        "new_customer_revenue": orders[orders["is_first_day_order"]].groupby("order_month")["revenue"].sum(),
        "returning_customer_revenue": orders[~orders["is_first_day_order"]].groupby("order_month")["revenue"].sum(),
    }).fillna(0)
    out["avg_order_value"] = out["revenue"] / out["orders"]
    cols = ["orders", "active_customers", "revenue", "avg_order_value", "new_customer_revenue", "returning_customer_revenue"]
    return out[cols].round(2).reset_index()


# ----------------------------------------------------------------------------
# 3. CUSTOMER TIMELINE + REPEAT (observability-adjusted)
# ----------------------------------------------------------------------------
def customer_timeline(orders: pd.DataFrame, cfg: Config, data_end: pd.Timestamp) -> pd.DataFrame:
    o = orders.assign(day=orders["order_date"].dt.normalize())
    first_day = o.groupby("customer_id")["day"].min().rename("first_day")
    o = o.join(first_day, on="customer_id")
    second_day = o[o["day"] > o["first_day"]].groupby("customer_id")["day"].min().rename("second_day")
    cust = pd.concat([first_day, second_day], axis=1)
    cust["days_to_repeat"] = (cust["second_day"] - cust["first_day"]).dt.days
    cust["ever_repeated"] = cust["second_day"].notna()
    # Only customers observable for the full window can fairly be judged on "repeated within window"
    cust["eligible"] = cust["first_day"] + pd.Timedelta(days=cfg.repeat_window_days) <= data_end
    cust["repeated_in_window"] = cust["days_to_repeat"].le(cfg.repeat_window_days).fillna(False)
    cust["cohort_month"] = cust["first_day"].dt.strftime("%Y-%m")
    return cust


def enrich_orders(orders: pd.DataFrame, cust: pd.DataFrame) -> pd.DataFrame:
    o = orders.join(cust[["first_day", "cohort_month"]], on="customer_id")
    o["order_day"] = o["order_date"].dt.normalize()
    o["is_first_day_order"] = o["order_day"] == o["first_day"]
    o["order_month"] = o["order_date"].dt.strftime("%Y-%m")
    o["order_mnum"] = o["order_date"].dt.year * 12 + o["order_date"].dt.month
    o["cohort_mnum"] = o["first_day"].dt.year * 12 + o["first_day"].dt.month
    o["month_index"] = o["order_mnum"] - o["cohort_mnum"]
    return o


# ----------------------------------------------------------------------------
# 4. COHORTS
# ----------------------------------------------------------------------------
def cohort_tables(orders, cohort_base_df, cust, data_start, data_end, cfg):
    last_complete = data_end.to_period("M") if data_end.day == data_end.days_in_month else data_end.to_period("M") - 1
    lc_num = last_complete.year * 12 + last_complete.month
    first_data_month = str(data_start.to_period("M"))
    h = cfg.value_horizon_months

    sizes = cust.groupby("cohort_month").size()
    cm = orders.drop_duplicates("cohort_month").set_index("cohort_month")["cohort_mnum"]

    o = orders.assign(repeat_revenue=orders["revenue"].where(~orders["is_first_day_order"], 0.0))
    agg = (o.groupby(["cohort_month", "month_index"])
             .agg(active_customers=("customer_id", "nunique"),
                  revenue=("revenue", "sum"),
                  repeat_revenue=("repeat_revenue", "sum")).reset_index())

    # Only keep cells that are OBSERVABLE (month fully elapsed). Unobserved != zero retention.
    rows = []
    for c, size in sizes.items():
        obs = lc_num - cm[c]
        for k in range(0, obs + 1):
            rows.append((c, k, size))
    cols = ["cohort_month", "month_index", "cohort_size", "active_customers", "revenue", "repeat_revenue"]
    if rows:
        grid = pd.DataFrame(rows, columns=["cohort_month", "month_index", "cohort_size"])
        long = grid.merge(agg, on=["cohort_month", "month_index"], how="left")
        long[["active_customers", "revenue", "repeat_revenue"]] = long[["active_customers", "revenue", "repeat_revenue"]].fillna(0)
        long["retention_rate"] = (long["active_customers"] / long["cohort_size"]).round(4)
        long["is_acquisition_cohort"] = long["cohort_month"] != first_data_month
        long = long.round({"revenue": 2, "repeat_revenue": 2})
    else:
        long = pd.DataFrame(columns=cols + ["retention_rate", "is_acquisition_cohort"])

    # ---- cohort summary: base totals + adjusted repeat + flags ----
    summ = cohort_base_df.copy()
    cust2 = cust.assign(rep_elig=cust["eligible"] & cust["repeated_in_window"])
    w = cust2.groupby("cohort_month").agg(eligible_customers=("eligible", "sum"),
                                          repeated_within_window=("rep_elig", "sum")).reset_index()
    summ = summ.merge(w, on="cohort_month", how="left")
    summ["repeat_rate_ever_unadjusted"] = (summ["ever_repeated_customers"] / summ["customers"]).round(3)
    summ[f"repeat_rate_{cfg.repeat_window_days}d_adjusted"] = np.where(
        summ["eligible_customers"] > 0, summ["repeated_within_window"] / summ["eligible_customers"].replace(0, np.nan), np.nan).round(3)
    summ["is_acquisition_cohort"] = summ["cohort_month"] != first_data_month
    summ["low_sample"] = summ["customers"] < cfg.min_cohort_size
    summ["fully_observed_window"] = summ["eligible_customers"] == summ["customers"]
    summ["note"] = np.where(~summ["is_acquisition_cohort"], "First month of data: includes pre-existing customers",
                    np.where(summ["low_sample"], "Small cohort: treat with caution", ""))

    # ---- 12-month observed value + estimated financials (kept separate) ----
    mature = [c for c in sizes.index if lc_num - cm[c] >= h - 1 and c != first_data_month]
    value = pd.DataFrame()
    if mature:
        sub = long[long["cohort_month"].isin(mature) & (long["month_index"] < h)]
        v = sub.groupby("cohort_month").agg(observed_revenue=("revenue", "sum"),
                                            observed_repeat_revenue=("repeat_revenue", "sum"))
        v["cohort_customers"] = sizes.reindex(v.index)
        v[f"OBSERVED_revenue_{h}m_per_customer"] = v["observed_revenue"] / v["cohort_customers"]
        v[f"OBSERVED_repeat_revenue_{h}m_per_customer"] = v["observed_repeat_revenue"] / v["cohort_customers"]
        v["OBSERVED_repeat_revenue_share"] = v["observed_repeat_revenue"] / v["observed_revenue"].replace(0, np.nan)
        v["ASSUMPTION_gross_margin"] = cfg.gross_margin
        v["ASSUMPTION_target_ltv_to_cac"] = cfg.target_ltv_cac
        gp = v[f"OBSERVED_revenue_{h}m_per_customer"] * cfg.gross_margin
        v[f"ESTIMATED_gross_profit_{h}m_per_customer"] = gp
        v["ESTIMATED_breakeven_cac_ceiling"] = gp
        v["ESTIMATED_max_cac_at_target_ratio"] = gp / cfg.target_ltv_cac
        v["low_sample"] = v["cohort_customers"] < cfg.min_cohort_size
        value = v.round(3).reset_index()

    # ---- sensitivity of the CAC ceiling to the assumptions ----
    sens = pd.DataFrame()
    if not value.empty:
        base = value[~value["low_sample"]]
        base = base if not base.empty else value
        med = base[f"OBSERVED_revenue_{h}m_per_customer"].median()
        sens = pd.DataFrame([{"assumed_gross_margin": m, "target_ltv_to_cac": t,
                              "median_observed_revenue_per_customer": round(med, 2),
                              "ESTIMATED_max_cac": round(med * m / t, 2)}
                             for m in (0.20, 0.30, 0.40) for t in (2, 3, 4)])
    return long, summ, value, sens, str(last_complete)


# ----------------------------------------------------------------------------
# 5. SEGMENTS
# ----------------------------------------------------------------------------
def build_segments(orders, df, cust, data_start, data_end, cfg):
    snapshot = data_end.normalize() + pd.Timedelta(days=1)
    days = (orders[["customer_id", "order_date"]]
            .assign(day=orders["order_date"].dt.normalize())
            .drop_duplicates(["customer_id", "day"]).sort_values(["customer_id", "day"]))
    days["gap"] = days.groupby("customer_id")["day"].diff().dt.days
    typical_gap = days.groupby("customer_id")["gap"].median().rename("typical_gap_days")
    all_gaps = days["gap"].dropna()
    p75_gap = float(all_gaps.quantile(0.75)) if len(all_gaps) else np.nan

    c = orders.groupby("customer_id").agg(orders=("invoice", "nunique"), revenue=("revenue", "sum"),
                                          first_order_date=("order_date", "min"), last_order_date=("order_date", "max"))
    c["purchase_days"] = days.groupby("customer_id")["day"].nunique()
    c["recency_days"] = (snapshot - c["last_order_date"].dt.normalize()).dt.days
    c = c.join(typical_gap)
    c["overdue_ratio"] = (c["recency_days"] / c["typical_gap_days"].replace(0, np.nan)).round(2)
    country = df.sort_values("invoice_date").groupby("customer_id")["country"].last()
    c = c.join(country).join(cust[["cohort_month", "eligible", "repeated_in_window", "ever_repeated"]])

    thr = c["revenue"].quantile(1 - cfg.high_value_pct)
    c["high_value"] = c["revenue"] >= thr
    active = c["recency_days"] <= cfg.inactive_days
    rep = c["purchase_days"] >= 2
    c["segment"] = np.select(
        [c["high_value"] & active, c["high_value"] & ~active, ~c["high_value"] & active & rep,
         ~c["high_value"] & active & ~rep, ~c["high_value"] & ~active & rep],
        [SEGMENTS[0], SEGMENTS[1], SEGMENTS[2], SEGMENTS[3], SEGMENTS[4]], default=SEGMENTS[5])
    c["recommended_action"] = c["segment"].map(SEGMENT_ACTIONS)
    c["revenue"] = c["revenue"].round(2)

    span_years = max((data_end - data_start).days / 365.25, 1.0)
    summ = c.groupby("segment").agg(customers=("revenue", "size"), total_revenue=("revenue", "sum"),
                                    avg_revenue=("revenue", "mean"), avg_purchase_days=("purchase_days", "mean"),
                                    avg_recency_days=("recency_days", "mean"))
    summ = summ.reindex(SEGMENTS)                      # every segment always present
    summ["customers"] = summ["customers"].fillna(0).astype(int)
    summ["total_revenue"] = summ["total_revenue"].fillna(0)
    summ = summ.fillna(0)
    summ["share_of_customers"] = summ["customers"] / max(summ["customers"].sum(), 1)
    summ["revenue_share"] = summ["total_revenue"] / max(summ["total_revenue"].sum(), 1e-9)
    summ["avg_annual_revenue_when_observed"] = summ["total_revenue"] / span_years / summ["customers"].replace(0, np.nan)
    summ["recommended_action"] = summ.index.map(SEGMENT_ACTIONS)
    summ = summ.round(3).reset_index().rename(columns={"index": "segment"})
    summ["revenue_rank"] = summ["total_revenue"].rank(ascending=False, method="min").astype(int)

    sens = pd.DataFrame([{
        "inactive_days_threshold": d,
        "high_value_customers_inactive": int((c["high_value"] & (c["recency_days"] > d)).sum()),
        "their_observed_revenue": round(float(c.loc[c["high_value"] & (c["recency_days"] > d), "revenue"].sum()), 2)}
        for d in (60, 90, 120, 180)])

    wb = (c[c["segment"] == "High-value at risk"].sort_values("revenue", ascending=False)
          .head(cfg.winback_list_size).reset_index()
          [["customer_id", "country", "revenue", "purchase_days", "last_order_date", "recency_days",
            "typical_gap_days", "overdue_ratio"]])
    return c.reset_index(), summ, sens, wb, p75_gap, span_years


# ----------------------------------------------------------------------------
# 6. CONCENTRATION + PRODUCTS
# ----------------------------------------------------------------------------
def concentration(seg):
    s = seg["revenue"].sort_values(ascending=False).reset_index(drop=True)
    n, total = len(s), s.sum()
    rows = [{"top_pct_customers": f"{int(p * 100)}%", "customers": max(1, int(n * p)),
             "revenue_share": round(s.iloc[:max(1, int(n * p))].sum() / total, 3)} for p in (0.01, 0.05, 0.10, 0.20, 0.50)]
    asc = np.sort(s.values)
    gini = float((2 * np.sum(np.arange(1, n + 1) * asc) / (n * asc.sum())) - (n + 1) / n) if n and asc.sum() > 0 else np.nan
    cum = s.cumsum() / total
    pct_for_half = float((cum < 0.5).sum() + 1) / n if n else np.nan
    curve = pd.DataFrame({"customer_pct": np.arange(1, n + 1) / n, "cum_revenue_share": cum.values})
    curve = curve.iloc[::max(1, n // 500)].round(4)
    return pd.DataFrame(rows), curve, gini, pct_for_half


def wilson(k, n, z=1.96):
    k, n = np.asarray(k, float), np.asarray(n, float)
    p = k / n
    d = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / d
    return centre - half, centre + half


def product_repeat(df, cust, cfg):
    empty = pd.DataFrame(columns=["stock_code", "description", "first_day_buyers", "repeaters", "repeat_rate",
                                  "baseline_repeat_rate", "lift", "lift_ci_low", "lift_ci_high", "confidence"])
    elig = cust[cust["eligible"]]
    if elig.empty:
        return empty, np.nan, cfg.min_product_customers
    baseline = float(elig["repeated_in_window"].mean())
    d = df[["customer_id", "stock_code", "invoice_date"]].copy()
    d["day"] = d["invoice_date"].dt.normalize()
    d = d.join(cust["first_day"], on="customer_id")
    d = d[(d["day"] == d["first_day"]) & d["customer_id"].isin(elig.index)]
    d = d.drop_duplicates(["customer_id", "stock_code"])
    d = d.join(elig["repeated_in_window"].rename("repeated"), on="customer_id")
    g = d.groupby("stock_code").agg(first_day_buyers=("customer_id", "nunique"), repeaters=("repeated", "sum"))
    used = cfg.min_product_customers
    for t in (cfg.min_product_customers, 50, 30):
        used = t
        sel = g[g["first_day_buyers"] >= t]
        if len(sel) >= 5:
            break
    if sel.empty or baseline == 0 or np.isnan(baseline):
        return empty, baseline, used
    sel = sel.copy()
    sel["repeat_rate"] = sel["repeaters"] / sel["first_day_buyers"]
    lo, hi = wilson(sel["repeaters"], sel["first_day_buyers"])
    sel["baseline_repeat_rate"] = baseline
    sel["lift"] = sel["repeat_rate"] / baseline
    sel["lift_ci_low"], sel["lift_ci_high"] = lo / baseline, hi / baseline
    sel["confidence"] = np.where(sel["lift_ci_low"] > 1, "clearly above average",
                          np.where(sel["lift_ci_high"] < 1, "clearly below average", "inconclusive"))
    descs = df[df["stock_code"].isin(sel.index)].groupby("stock_code")["description"].agg(
        lambda x: x.mode().iat[0] if len(x.mode()) else "")
    sel["description"] = sel.index.map(descs)
    sel = sel.sort_values("lift", ascending=False).reset_index().round(3)
    return sel[empty.columns], baseline, used


# ----------------------------------------------------------------------------
# 7. RECOMMENDATIONS (hypotheses, not findings)
# ----------------------------------------------------------------------------
def gbp(x):
    return f"GBP {x:,.0f}"


def build_recommendations(seg_summ, conc, prod, base_rate, cfg, span_years):
    s = seg_summ.set_index("segment")
    top10 = conc.loc[conc["top_pct_customers"] == "10%", "revenue_share"].iloc[0]
    recs = []

    def row(prio, action, segname, evidence, detail, hypothesis, metric, test):
        n = int(s.loc[segname, "customers"])
        if n == 0:
            return
        recs.append(dict(priority=prio, action=action, target_segment=segname, customers=n,
                         evidence_from_data=evidence, proposed_activity=detail,
                         hypothesis_untested=hypothesis, success_metric=metric, test_design=test))

    r = s.loc["High-value at risk"]
    row(1, "WIN BACK", "High-value at risk",
        f"{int(r.customers):,} high-value customers ({r.share_of_customers:.1%} of customers) account for "
        f"{gbp(r.total_revenue)} ({r.revenue_share:.0%}) of observed revenue but have not bought for {cfg.inactive_days}+ days.",
        "Personal outreach (account-manager call or tailored email) with a reason to return; prioritise by revenue and by how overdue "
        "each customer is versus their own purchase rhythm (see winback_priority_list.csv).",
        "HYPOTHESIS: contacted customers will reactivate at a higher rate than a no-contact holdout, and the incremental gross profit "
        "will exceed campaign cost. Not tested - the dataset contains no campaign data.",
        "Incremental reactivation rate (contacted minus holdout) within 60 days; incremental gross profit per contacted customer minus cost per contact.",
        "Randomly hold out 20% of the list (no contact). Compare purchases over 60 days after launch. Pre-agree the minimum uplift needed to justify the cost.")

    r = s.loc["Champions"]
    row(2, "RETAIN", "Champions",
        f"{int(r.customers):,} Champions generate {r.revenue_share:.0%} of observed revenue. Across all customers, the top 10% of spenders "
        f"generate {top10:.0%}, so losing a few key accounts would materially affect revenue.",
        "Protect rather than discount: named contact, early access to new lines, proactive re-order reminders when a customer's usual gap is exceeded.",
        "HYPOTHESIS: proactive contact when a Champion becomes 'overdue' reduces their churn into the at-risk segment. Not tested.",
        "Share of Champions still active after 6 months; revenue retention of the top-decile customer group.",
        "Trigger outreach when recency exceeds 1.5x the customer's own median purchase gap; compare with customers not triggered (random split).")

    r = s.loc["Recent one-time buyers"]
    strong = prod[prod["confidence"] == "clearly above average"].head(3) if len(prod) else prod
    if len(strong):
        prod_txt = ("Products first bought by customers who repeat more than average (association only): "
                    + "; ".join(f"{p.description} ({p.lift:.2f}x)" for p in strong.itertuples()) + ".")
    else:
        prod_txt = "No product shows a statistically clear repeat-purchase lift, so a second-purchase campaign should NOT be built around specific products."
    row(3, "GROW", "Recent one-time buyers",
        f"{int(r.customers):,} recent customers have bought once. The window-adjusted baseline repeat rate is "
        f"{base_rate:.0%} within {cfg.repeat_window_days} days." if not np.isnan(base_rate) else
        f"{int(r.customers):,} recent customers have bought once.",
        f"Targeted second-purchase email/offer shortly after the first order. {prod_txt}",
        "HYPOTHESIS: a well-timed second-purchase prompt raises the share of new customers who repeat within the window. Product association is not causation.",
        f"{cfg.repeat_window_days}-day repeat rate of treated vs control new customers.",
        "A/B test on new customers: random 50/50 split, same window, same definition of repeat as this analysis.")

    r = s.loc["Lapsed repeat buyers"]
    row(4, "WIN BACK (low cost)", "Lapsed repeat buyers",
        f"{int(r.customers):,} customers bought on 2+ days previously but are inactive; observed revenue {gbp(r.total_revenue)}.",
        "Automated, low-cost email sequence only; avoid expensive incentives because individual value is lower.",
        "HYPOTHESIS: automated reactivation emails are profitable at low cost per contact. Not tested.",
        "Reactivation rate and incremental gross profit per email sent versus holdout.", "20% holdout, 60-day window.")

    r = s.loc["Lost one-time buyers"]
    row(5, "DEPRIORITISE", "Lost one-time buyers",
        f"{int(r.customers):,} customers bought once and have been inactive for {cfg.inactive_days}+ days; average observed revenue {gbp(r.avg_revenue)}.",
        "No paid win-back spend; keep on low-cost lists only.",
        "HYPOTHESIS: spend is better directed at the segments above. Not tested.",
        "Cost per reactivated customer versus other segments.", "Small test only if budget allows.")
    return pd.DataFrame(recs)


def winback_scenarios(seg_summ, cfg, span_years):
    r = seg_summ.set_index("segment").loc["High-value at risk"]
    annual = r["total_revenue"] / span_years
    return pd.DataFrame([{
        "label": "ILLUSTRATIVE SCENARIO - not a forecast",
        "reactivation_rate_assumed": rate, "share_of_past_annual_spend_retained_assumed": 0.5,
        "customers_at_risk": int(r["customers"]),
        "past_average_annual_revenue_of_segment": round(annual, 0),
        "incremental_revenue_12m": round(annual * rate * 0.5, 0),
        "ESTIMATED_incremental_gross_profit_12m": round(annual * rate * 0.5 * cfg.gross_margin, 0),
        "assumed_gross_margin": cfg.gross_margin} for rate in (0.05, 0.10, 0.20)])


# ----------------------------------------------------------------------------
# 8. KPIs + REPORT
# ----------------------------------------------------------------------------
def md_table(df, cols=None):
    if df is None or len(df) == 0:
        return "_No rows available._"
    d = df[cols] if cols else df
    head = "| " + " | ".join(map(str, d.columns)) + " |\n|" + "---|" * len(d.columns) + "\n"
    return head + "\n".join("| " + " | ".join(str(v) for v in r) + " |" for r in d.itertuples(index=False))


def write_report(out, cfg, dq, kpis, cohort_summ, value, sens_fin, seg_summ, seg_sens, conc, gini, pct_half,
                 prod, base_rate, prod_threshold, recs, scen, p75_gap, data_start, data_end, last_complete):
    k = kpis.set_index("metric")["value"]
    w = cfg.repeat_window_days
    L = ["# Profitable Customer Growth - Auto-Generated Analysis Report",
         f"*Data: {data_start:%d %b %Y} to {data_end:%d %b %Y}. Observed currency: GBP. Last complete month used for cohorts: {last_complete}.*", "",
         "## 1. Data quality", md_table(dq), "",
         "## 2. Headline KPIs (OBSERVED from data)",
         f"- Revenue: **{gbp(k['total_revenue'])}** from **{int(k['orders']):,}** orders and **{int(k['customers']):,}** customers",
         f"- Average order value: **GBP {k['avg_order_value']:,.2f}** | Revenue per customer: **GBP {k['revenue_per_customer']:,.2f}**",
         f"- Repeat-customer rate, unadjusted (ever bought on a later day): **{k['repeat_rate_unadjusted']:.1%}**"]
    if not np.isnan(k[f"repeat_rate_{w}d_adjusted"]):
        L.append(f"- Repeat-customer rate, **adjusted** (repeat within {w} days, only customers observable for {w}+ days; "
                 f"n={int(k['eligible_customers']):,}): **{k[f'repeat_rate_{w}d_adjusted']:.1%}**")
    else:
        L.append(f"- Adjusted repeat rate not available: no customer has {w}+ days of history.")
    L += [f"- Revenue from repeat orders: **{k['repeat_revenue_share']:.0%}** of total", ""]

    L += ["## 3. Cohorts (acquisition month)"]
    acq = cohort_summ[cohort_summ["is_acquisition_cohort"] & ~cohort_summ["low_sample"]]
    adj_col = f"repeat_rate_{w}d_adjusted"
    if len(acq):
        full = acq[acq["fully_observed_window"]]
        if len(full):
            b = full.sort_values(adj_col, ascending=False).iloc[0]
            L.append(f"- Highest {w}-day repeat rate among fully observed cohorts: **{b['cohort_month']}** ({b[adj_col]:.0%}, {int(b['customers'])} customers).")
        if len(value):
            h = cfg.value_horizon_months
            col = f"OBSERVED_repeat_revenue_{h}m_per_customer"
            b = value[~value["low_sample"]].sort_values(col, ascending=False)
            if len(b):
                b = b.iloc[0]
                L.append(f"- Highest observed {h}-month repeat revenue per customer: **{b['cohort_month']}** (GBP {b[col]:,.2f} per customer). "
                         "All cohorts are compared over the same 12-month window.")
        L.append("- The first month of the dataset is excluded from acquisition comparisons because it includes pre-existing customers; "
                 "cohorts with fewer customers than the minimum are flagged `low_sample`.")
    else:
        L.append("- Not enough cohort history for reliable cohort comparisons.")
    L.append("")

    L += ["## 4. Segments and the 'which customers should we prioritise?' answer",
          md_table(seg_summ.assign(total_revenue=seg_summ["total_revenue"].round(0), avg_recency_days=seg_summ["avg_recency_days"].round(0),
                                   revenue_share=(seg_summ["revenue_share"] * 100).round(1).astype(str) + "%"),
                   ["segment", "recommended_action", "customers", "total_revenue", "revenue_share", "avg_recency_days"]),
          "",
          f"Inactivity threshold: **{cfg.inactive_days} days**. "
          + (f"For reference, 75% of repeat purchase gaps in this data are {p75_gap:.0f} days or shorter." if not np.isnan(p75_gap) else "")
          + " Sensitivity to this threshold:", md_table(seg_sens), ""]

    L += ["## 5. Revenue concentration",
          md_table(conc),
          f"- The top **{pct_half:.0%}** of customers generate half of all revenue. Gini coefficient: **{gini:.2f}** (0 = equal, 1 = one customer).", ""]

    L += [f"## 6. Products associated with repeat purchasing",
          f"Method: among customers observable for {w}+ days, share who repeated within {w} days after first buying each product on their first purchase day. "
          f"Baseline: **{base_rate:.1%}**. Minimum {prod_threshold} first-day buyers per product. **Association, not causation** - "
          "bulk/wholesale buyers may drive both product choice and repeat behaviour." if not np.isnan(base_rate) else "Not enough data.",
          md_table(prod.head(10), ["description", "first_day_buyers", "repeat_rate", "lift", "lift_ci_low", "lift_ci_high", "confidence"]), ""]

    L += ["## 7. Financial view - OBSERVED vs ESTIMATED (kept separate)",
          "**Observed** (calculated directly from transactions): revenue, orders, repeat revenue, revenue per customer.", "",
          f"**Estimated** (depends on assumptions NOT in the data): gross margin = {cfg.gross_margin:.0%}, target LTV:CAC = {cfg.target_ltv_cac:.0f}:1. "
          "The dataset contains no advertising spend and no product cost, so **actual CAC, actual profit and actual LTV:CAC cannot be measured**. "
          "The figures below are a *ceiling*: the most the business could pay to acquire a customer at that margin and target ratio."]
    if len(value):
        h = cfg.value_horizon_months
        cols = ["cohort_month", "cohort_customers", f"OBSERVED_revenue_{h}m_per_customer",
                f"ESTIMATED_gross_profit_{h}m_per_customer", "ESTIMATED_max_cac_at_target_ratio"]
        L += [md_table(value, cols), "", "Sensitivity of the estimated CAC ceiling to the assumptions:", md_table(sens_fin)]
    else:
        L.append(f"No cohort has {cfg.value_horizon_months} complete months of history in this dataset, so no value estimate is shown.")
    L.append("")

    L += ["## 8. Recommendations (HYPOTHESES to test, not proven results)",
          "Each recommendation states the evidence from the data, the proposed activity, the untested hypothesis, how success would be measured, and how it should be tested.", ""]
    for r in recs.itertuples():
        L += [f"### {r.priority}. {r.action}: {r.target_segment} ({r.customers:,} customers)",
              f"- **Evidence:** {r.evidence_from_data}", f"- **Proposed activity:** {r.proposed_activity}",
              f"- **{r.hypothesis_untested}**", f"- **Success metric:** {r.success_metric}", f"- **Test design:** {r.test_design}", ""]
    if len(scen):
        L += ["**Illustrative win-back scenario** (assumed reactivation rates; spend after reactivation assumed at 50% of past average annual spend; not a forecast):",
              md_table(scen, ["reactivation_rate_assumed", "incremental_revenue_12m", "ESTIMATED_incremental_gross_profit_12m"]), ""]

    L += ["## 9. Limitations",
          "- No advertising spend or product costs: CAC and profit are assumption-based ceilings, not measurements.",
          "- Only customers with a Customer ID are analysed; guest orders are excluded (see data quality table).",
          "- The retailer largely sells to wholesale (B2B) customers, so results may not generalise to consumer retail.",
          "- Segments are a snapshot as of the last date in the data. The dataset ends part-way through its last month.",
          "- Observational data: product and segment patterns show association, not causal effects."]
    (out / "report.md").write_text("\n".join(L), encoding="utf-8")


def build_kpis(orders, cust, seg, conc, gini, cfg):
    w = cfg.repeat_window_days
    elig = cust[cust["eligible"]]
    tot = orders["revenue"].sum()
    rep_rev = orders.loc[~orders["is_first_day_order"], "revenue"].sum()
    top10 = conc.loc[conc["top_pct_customers"] == "10%", "revenue_share"].iloc[0]
    rows = [("total_revenue", tot, "Sum of quantity x price (cleaned data)"),
            ("orders", orders["invoice"].nunique(), "Distinct invoices"),
            ("customers", orders["customer_id"].nunique(), "Distinct customers with a Customer ID"),
            ("avg_order_value", tot / orders["invoice"].nunique(), "Revenue / orders"),
            ("revenue_per_customer", tot / orders["customer_id"].nunique(), "Revenue / customers (over whole period)"),
            ("repeat_rate_unadjusted", cust["ever_repeated"].mean(), "Share of customers with an order on a later day. Biased down for recent customers"),
            (f"repeat_rate_{w}d_adjusted", elig["repeated_in_window"].mean() if len(elig) else np.nan,
             f"Share repeating within {w} days, among customers observable for {w}+ days"),
            ("eligible_customers", len(elig), f"Customers with >= {w} days of history"),
            ("repeat_revenue_share", rep_rev / tot, "Revenue from orders placed on a later day than the first purchase day"),
            ("top10pct_revenue_share", top10, "Share of revenue from top 10% of customers"),
            ("gini_coefficient", gini, "Revenue inequality across customers")]
    return pd.DataFrame(rows, columns=["metric", "value", "definition"]).assign(type="OBSERVED")


# ----------------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------------
def run_pipeline(raw: pd.DataFrame, cfg: Config, out_dir: str | Path = "output") -> dict:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    df, dq = clean(standardise_columns(raw))
    data_start, data_end = df["invoice_date"].min(), df["invoice_date"].max()
    print(dq.to_string(index=False))

    orders_raw = build_orders(df)
    cust = customer_timeline(orders_raw, cfg, data_end)
    orders = enrich_orders(orders_raw, cust)
    cohort_base_df = cohort_base(orders)
    monthly = monthly_kpis(orders)

    long, cohort_summ, value, sens_fin, last_complete = cohort_tables(orders, cohort_base_df, cust, data_start, data_end, cfg)
    seg, seg_summ, seg_sens, winback, p75_gap, span_years = build_segments(orders, df, cust, data_start, data_end, cfg)
    conc, curve, gini, pct_half = concentration(seg)
    prod, base_rate, prod_threshold = product_repeat(df, cust, cfg)
    kpis = build_kpis(orders, cust, seg, conc, gini, cfg)
    recs = build_recommendations(seg_summ, conc, prod, base_rate, cfg, span_years)
    scen = winback_scenarios(seg_summ, cfg, span_years)

    monthly["is_partial_month"] = monthly["order_month"] > last_complete
    fact_orders = orders[["customer_id", "invoice", "order_date", "order_month", "cohort_month", "revenue", "is_first_day_order"]]
    dim_customer = seg.drop(columns=["eligible", "repeated_in_window", "ever_repeated"]).merge(
        cust.reset_index()[["customer_id", "eligible", "repeated_in_window", "ever_repeated", "days_to_repeat"]], on="customer_id")
    dim_customer = dim_customer.rename(columns={"eligible": "eligible_for_repeat_window"})

    files = {
        "fact_orders.csv": fact_orders, "dim_customer.csv": dim_customer, "kpis.csv": kpis, "monthly_kpis.csv": monthly,
        "cohort_retention_long.csv": long, "cohort_summary.csv": cohort_summ, "cohort_value_12m.csv": value,
        "cac_ceiling_sensitivity.csv": sens_fin, "segment_summary.csv": seg_summ, "segment_threshold_sensitivity.csv": seg_sens,
        "winback_priority_list.csv": winback, "revenue_concentration.csv": conc, "pareto_curve.csv": curve,
        "product_repeat_lift.csv": prod, "recommendations.csv": recs, "winback_scenarios.csv": scen, "data_quality.csv": dq}
    for name, frame in files.items():
        frame.to_csv(out / name, index=False)
    write_report(out, cfg, dq, kpis, cohort_summ, value, sens_fin, seg_summ, seg_sens, conc, gini, pct_half, prod, base_rate,
                 prod_threshold, recs, scen, p75_gap, data_start, data_end, last_complete)
    print(f"\nDone. {len(files)} CSVs + report.md written to {out.resolve()}")
    return {"kpis": kpis, "cohort_summary": cohort_summ, "segments": seg_summ, "products": prod, "files": list(files)}


def main():
    ap = argparse.ArgumentParser(description="Profitable Customer Growth Analysis")
    ap.add_argument("path", help="Path to online_retail_II.xlsx (or .csv)")
    ap.add_argument("--out", default="output")
    ap.add_argument("--margin", type=float, default=0.30, help="ASSUMED gross margin (default 0.30)")
    ap.add_argument("--target-ratio", type=float, default=3.0, help="Target LTV:CAC for the CAC ceiling (default 3)")
    ap.add_argument("--inactive-days", type=int, default=90)
    ap.add_argument("--repeat-window", type=int, default=90)
    a = ap.parse_args()
    cfg = Config(gross_margin=a.margin, target_ltv_cac=a.target_ratio, inactive_days=a.inactive_days, repeat_window_days=a.repeat_window)
    run_pipeline(load(a.path), cfg, a.out)


if __name__ == "__main__":
    main()

Writing customer_growth_analysis.py


In [3]:
import os

print("File exists:", os.path.exists("customer_growth_analysis.py"))
print("File size:", os.path.getsize("customer_growth_analysis.py"), "bytes")

File exists: True
File size: 40352 bytes


In [7]:
%run customer_growth_analysis.py "/Users/dipachowdhury/Downloads/online_retail_II.xlsx"

                                           step  rows_removed  rows_remaining
                                       Raw rows             0         1067371
Missing / invalid Customer ID (guest checkouts)        243007          824364
         Cancellation invoices (start with 'C')         18744          805620
                 Non-positive quantity or price            71          805549
 Non-product codes (postage, fees, adjustments)          2645          802904
                       Unparseable invoice date             0          802904
                           Exact duplicate rows         26060          776844

Done. 17 CSVs + report.md written to /Users/dipachowdhury/output


In [9]:
import os

output_folder = "/Users/dipachowdhury/output"

for file in sorted(os.listdir(output_folder)):
    print(file)

cac_ceiling_sensitivity.csv
cohort_retention_long.csv
cohort_summary.csv
cohort_value_12m.csv
data_quality.csv
dim_customer.csv
fact_orders.csv
kpis.csv
monthly_kpis.csv
pareto_curve.csv
product_repeat_lift.csv
recommendations.csv
report.md
revenue_concentration.csv
segment_summary.csv
segment_threshold_sensitivity.csv
winback_priority_list.csv
winback_scenarios.csv


In [11]:
report_path = os.path.join(output_folder, "report.md")

with open(report_path, "r", encoding="utf-8") as f:
    report = f.read()

print(report)

# Profitable Customer Growth - Auto-Generated Analysis Report
*Data: 01 Dec 2009 to 09 Dec 2011. Observed currency: GBP. Last complete month used for cohorts: 2011-11.*

## 1. Data quality
| step | rows_removed | rows_remaining |
|---|---|---|
| Raw rows | 0 | 1067371 |
| Missing / invalid Customer ID (guest checkouts) | 243007 | 824364 |
| Cancellation invoices (start with 'C') | 18744 | 805620 |
| Non-positive quantity or price | 71 | 805549 |
| Non-product codes (postage, fees, adjustments) | 2645 | 802904 |
| Unparseable invoice date | 0 | 802904 |
| Exact duplicate rows | 26060 | 776844 |

## 2. Headline KPIs (OBSERVED from data)
- Revenue: **GBP 17,081,129** from **36,600** orders and **5,853** customers
- Average order value: **GBP 466.70** | Revenue per customer: **GBP 2,918.35**
- Repeat-customer rate, unadjusted (ever bought on a later day): **71.4%**
- Repeat-customer rate, **adjusted** (repeat within 90 days, only customers observable for 90+ days; n=5,257): **44.2%**
- Rev